In [1]:
# 1️⃣ First, upgrade huggingface_hub to a compatible version
!pip install -U "huggingface_hub>=0.28.0"

# 2️⃣ Reinstall PEFT to ensure it builds against the correct version
!pip install -U peft

# 3️⃣ (Optional) Reinstall transformers to align versions
!pip install -U transformers
!pip install -U torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# 2️⃣ Ensure all Hugging Face core libs align
!pip install -U "transformers==4.45.2" "huggingface_hub>=0.28.0" "peft==0.13.2" "accelerate==0.34.2" "datasets==2.19.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.8/503.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.0.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 1.0.1
    Uninstalling huggingface-hub-1.0.1:
      Successfully uninstalled huggingface-hub-1.0.1
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!unzip "/content/drive/MyDrive/Legal Summry/CLEANED TEXTS.zip" -d "/content/Cleaned/"

Archive:  /content/drive/MyDrive/Legal Summry/CLEANED TEXTS.zip
   creating: /content/Cleaned/Cleaned_Texts/
  inflating: /content/Cleaned/__MACOSX/._Cleaned_Texts  
  inflating: /content/Cleaned/Cleaned_Texts/.DS_Store  
  inflating: /content/Cleaned/__MACOSX/Cleaned_Texts/._.DS_Store  
   creating: /content/Cleaned/Cleaned_Texts/Fundamental Rights/
  inflating: /content/Cleaned/__MACOSX/Cleaned_Texts/._Fundamental Rights  
   creating: /content/Cleaned/Cleaned_Texts/Civil Appeal/
  inflating: /content/Cleaned/__MACOSX/Cleaned_Texts/._Civil Appeal  
  inflating: /content/Cleaned/Cleaned_Texts/Fundamental Rights/SC_FR_08_2019.txt  
  inflating: /content/Cleaned/__MACOSX/Cleaned_Texts/Fundamental Rights/._SC_FR_08_2019.txt  
  inflating: /content/Cleaned/Cleaned_Texts/Fundamental Rights/SC_FR_452_2019.txt  
  inflating: /content/Cleaned/__MACOSX/Cleaned_Texts/Fundamental Rights/._SC_FR_452_2019.txt  
  inflating: /content/Cleaned/Cleaned_Texts/Fundamental Rights/SC_FR_52_2015.txt  
  in

In [2]:
!unzip "/content/drive/MyDrive/Legal Summry/nllb_sinhala2english_lora.zip" -d "/content/adapter/"

Archive:  /content/drive/MyDrive/Legal Summry/nllb_sinhala2english_lora.zip
   creating: /content/adapter/nllb_sinhala2english_lora/
  inflating: /content/adapter/__MACOSX/._nllb_sinhala2english_lora  
  inflating: /content/adapter/nllb_sinhala2english_lora/adapter_model.safetensors  
  inflating: /content/adapter/__MACOSX/nllb_sinhala2english_lora/._adapter_model.safetensors  
  inflating: /content/adapter/nllb_sinhala2english_lora/tokenizer_config.json  
  inflating: /content/adapter/__MACOSX/nllb_sinhala2english_lora/._tokenizer_config.json  
  inflating: /content/adapter/nllb_sinhala2english_lora/special_tokens_map.json  
  inflating: /content/adapter/__MACOSX/nllb_sinhala2english_lora/._special_tokens_map.json  
  inflating: /content/adapter/nllb_sinhala2english_lora/sentencepiece.bpe.model  
  inflating: /content/adapter/__MACOSX/nllb_sinhala2english_lora/._sentencepiece.bpe.model  
  inflating: /content/adapter/nllb_sinhala2english_lora/tokenizer.json  
  inflating: /content/ada

## Restart Session after this

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import re
from peft import LoraConfig, get_peft_model, PeftModel
from safetensors.torch import load_file
import pandas as pd
import os
from pathlib import Path

## Extraction Functions

In [4]:
# Sinhala detection
SINHALA_PATTERN = re.compile(r'[\u0D80-\u0DFF]')

# QA markers
Q_MARKERS = ['Q:', 'Q.', 'පු:']
A_MARKERS = ['A:', 'A.', 'උ:','c:','C:']
ALL_MARKERS = Q_MARKERS + A_MARKERS

def clean_marker(text):
    """
    Normalize QA markers:
    - 'පු:' and 'Q.', 'Q:' → 'Q:'
    - 'උ:' and 'A.', 'A:' → 'A:'
    """
    text = text.strip()

    for q_marker in ['පු:', 'Q.', 'Q:']:
        if text.startswith(q_marker):
            text = text[len(q_marker):].strip()
            return 'Q:' + text

    for a_marker in ['A:', 'A.', 'උ:','c:','C:']:
        if text.startswith(a_marker):
            text = text[len(a_marker):].strip()
            return 'A:' + text

    return text

def extract_sinhala_block(text, lookahead=10):
    """
    Extract Sinhala text block from given text.
    """
    match = re.search(SINHALA_PATTERN, text)
    if not match:
        return ""

    text = text[match.start():]
    result = []
    i = 0
    while i < len(text):
        result.append(text[i])
        if text[i] in ['.', '!', '?', '”', '"', '\n']:
            snippet = text[i+1:i+1+lookahead]
            if not re.search(SINHALA_PATTERN, snippet):
                break
        i += 1

    block = ''.join(result).rstrip()
    while block and not SINHALA_PATTERN.search(block[-1]):
        block = block[:-1]
    return block.strip()

def extract_qa_with_signatures(text, signature_map=None, sig_prefix="§SIG", start_counter=1):
    """
    Extract Sinhala QA blocks, replace them in text with signatures, and update signature map.

    Args:
        text (str): Original text with English + Sinhala QA
        signature_map (dict): Existing signature map (updated in-place)
        sig_prefix (str): Prefix for signatures
        start_counter (int): Starting signature number

    Returns:
        processed_text (str): Text with Sinhala replaced by signatures
        signature_map (dict): Updated mapping signature -> Sinhala
        next_sig_counter (int): Next counter value for further signatures
    """
    signature_map = signature_map or {}
    sig_counter = start_counter

    qa_pairs = []

    pattern = r'(' + '|'.join(re.escape(m) for m in ALL_MARKERS) + r')'
    markers = [(m.group(), m.start()) for m in re.finditer(pattern, text)]

    if not markers:
        return text, signature_map, sig_counter

    current_q_sig = None
    processed_text = text

    for i, (marker, pos) in enumerate(markers):
        end = markers[i+1][1] if i+1 < len(markers) else len(text)
        block = text[pos:end].strip()
        normalized_block = clean_marker(block)
        sinhala_text = extract_sinhala_block(normalized_block)

        if not sinhala_text:
            continue

        if normalized_block.startswith("Q:") and not sinhala_text.endswith("?"):
            sinhala_text += "?"
        elif normalized_block.startswith("A:") and not sinhala_text.endswith("."):
            sinhala_text += "."

        # Assign signature
        sig = f"{sig_prefix}{sig_counter:04d}"
        signature_map[sig] = sinhala_text
        sig_counter += 1

        # Replace Sinhala in normalized block with signature
        new_block = normalized_block.replace(sinhala_text, sig)
        processed_text = processed_text.replace(block, new_block, 1)

        # Track QA pairs using signatures
        if marker in Q_MARKERS:
            current_q_sig = sig
        elif marker in A_MARKERS and current_q_sig:
            qa_pairs.append({'Q': current_q_sig, 'A': sig})
            current_q_sig = None

    return processed_text, signature_map, sig_counter

## Translation Functions

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

def translate_sinhala_to_english(signature_map, legal_dict=None, max_tokens=1024):
    """
    Translates Sinhala texts in a dictionary to English using LoRA-adapted NLLB-200 600M,
    with dictionary fallback, token-limit-aware chunking, and safe GPU inference.

    Args:
        signature_map (dict): {signature: sinhala_text}
        legal_dict (dict, optional): {sinhala_phrase: english_translation} to prioritize
        max_tokens (int): maximum token length per chunk
    Returns:
        dict: {signature: english_translation}
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv("/content/drive/MyDrive/Legal Summry/sin-eng.csv")
    legal_dict = dict(zip(df['sinhala'], df['english']))
    # Load model + tokenizer
    base_model_name = "facebook/nllb-200-distilled-600M"
    adapter_dir = "/content/adapter/nllb_sinhala2english_lora"

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)
    model.to(device)
    model.eval()

    tgt_lang = "eng_Latn"  # forced target language
    translated_map = {}

    for sig, sinhala_text in signature_map.items():

        # 1️⃣ Full-text dictionary lookup
        if legal_dict and sinhala_text in legal_dict:
            translated_map[sig] = legal_dict[sinhala_text]
            continue

        # 2️⃣ Tokenize and check length
        tokens = tokenizer(sinhala_text, return_tensors="pt", truncation=False).input_ids[0]
        if len(tokens) <= max_tokens:
            chunks = [sinhala_text]
        else:
            # Split by sentences
            sentences = sinhala_text.split("।")
            chunks = []
            current_chunk = ""

            for sentence in sentences:
                candidate = (current_chunk + " " + sentence).strip()
                candidate_tokens = tokenizer(candidate, return_tensors="pt").input_ids[0]

                if len(candidate_tokens) > max_tokens:
                    # Current chunk is ready to append
                    if current_chunk:
                        chunks.append(current_chunk.strip())

                    # Check if sentence itself is too long
                    sentence_tokens = tokenizer(sentence, return_tensors="pt").input_ids[0]
                    if len(sentence_tokens) > max_tokens:
                        # Split sentence into sub-chunks of max_tokens
                        for i in range(0, len(sentence_tokens), max_tokens):
                            sub_tokens = sentence_tokens[i:i+max_tokens]
                            sub_chunk = tokenizer.decode(sub_tokens, skip_special_tokens=True)
                            chunks.append(sub_chunk.strip())
                        current_chunk = ""
                    else:
                        current_chunk = sentence
                else:
                    current_chunk = candidate

            if current_chunk:
                chunks.append(current_chunk.strip())

        # 3️⃣ Translate each chunk with dictionary replacement
        translated_chunks = []
        for chunk in chunks:
            if legal_dict:
                for sin_phrase, eng_translation in legal_dict.items():
                    if sin_phrase in chunk:
                        chunk = chunk.replace(sin_phrase, eng_translation)

            # Model inference
            with torch.no_grad():
                inputs = tokenizer(
                    chunk,
                    return_tensors="pt",
                    truncation=True,
                    padding=True,
                    max_length=max_tokens
                ).to(device)
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                    max_length=256
                )
                english_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                translated_chunks.append(english_text)

            torch.cuda.empty_cache()  # prevent memory buildup

        # 4️⃣ Combine all chunks
        translated_map[sig] = " ".join(translated_chunks).strip()

    return translated_map

def replace_signatures_with_translations(translated_text, translated_map):
    """
    Replaces signature placeholders in a translated text with their English translations.

    Args:
        translated_text (str): Text containing signatures like §SIG0001
        translated_map (dict): {signature: english_translation}

    Returns:
        str: Text with signatures replaced by their translated context
    """

    # Regex to match §SIGxxxx patterns
    pattern = r"§SIG\d{4}"

    def replacer(match):
        sig = match.group(0)
        return translated_map.get(sig, sig)

    return re.sub(pattern, replacer, translated_text)


## Process Directory

In [6]:
# Sinhala character detection pattern
SINHALA_PATTERN = re.compile(r'[\u0D80-\u0DFF]')

def process_text_file(input_path, output_dir):
    """
    Process a Sinhala+English text file, extract all Sinhala content,
    replace with signatures, translate Sinhala parts, and save results.
    """

    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"\n📄 Processing file: {os.path.basename(input_path)}")
    signature_map = {}
    sig_counter = 1

    # QA extractor
    processed_text, signature_map, sig_counter = extract_qa_with_signatures(text)

    # Remaining Sinhala blocks
    text = processed_text
    i = 0
    n = len(text)
    while i < n:
        if text[i] == '§':
            end_sig = text.find(' ', i)
            if end_sig == -1:
                end_sig = n
            i = end_sig
            continue
        if SINHALA_PATTERN.match(text[i]):
            remaining_text = text[i:]
            block = extract_sinhala_block(remaining_text)
            if block:
                sig = f"§SIG{sig_counter:04d}"
                signature_map[sig] = block
                text = text[:i] + sig + text[i+len(block):]
                sig_counter += 1
                i += len(sig)
                n = len(text)
                continue
        i += 1

    processed_text = text

    # Step 3️⃣ — Translate Sinhala map to English
    translated_map = translate_sinhala_to_english(signature_map)

    # Step 4️⃣ — Replace all signatures with English translations
    final_translation = replace_signatures_with_translations(processed_text, translated_map)

    base_name = os.path.splitext(os.path.basename(input_path))[0]
    translated_output = os.path.join(output_dir, f"{base_name}_translated.txt")

    with open(translated_output, "w", encoding="utf-8") as f:
        f.write(final_translation)

    print(f"✅ Translated file saved → {translated_output}")
    print(f"🧭 Total Sinhala blocks: {len(signature_map)}")

    return signature_map

def process_directory(input_dir, output_dir, recursive=True):
    """
    Process all text files in a directory through the Sinhala translation pipeline.

    Args:
        input_dir (str): Path to input directory containing .txt files
        output_dir (str): Path to output directory where results will be saved
        recursive (bool): If True, process subdirectories as well
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    files = list(input_dir.rglob("*.txt")) if recursive else list(input_dir.glob("*.txt"))

    count = 0
    skipped = 0
    for file_path in files:
        # Compute relative path and target output directory
        rel_path = file_path.relative_to(input_dir).parent
        target_output_dir = output_dir / rel_path
        os.makedirs(target_output_dir, exist_ok=True)

        # Compute translated output file path
        base_name = file_path.stem
        translated_output_file = target_output_dir / f"{base_name}_translated.txt"

        # Skip if already exists
        if translated_output_file.exists():
            print(f"⏭️ Skipping already translated file: {file_path.name}")
            skipped += 1
            continue

        # Otherwise, try processing
        try:
            process_text_file(file_path, target_output_dir)
            count += 1
        except Exception as e:
            print(f"⚠️ Error processing {file_path.name}: {e}")

    print(f"\n✅ Completed processing {count} file(s) from {input_dir}")
    if skipped > 0:
        print(f"⏭️ Skipped {skipped} file(s) that were already translated.")


input_dir = "/content/Cleaned/Cleaned_Texts"
output_dir = "/content/drive/MyDrive/Legal Summry/Translated"
os.makedirs(output_dir, exist_ok = True)
process_directory(input_dir, output_dir)

⏭️ Skipping already translated file: SC_FR_139_2016.txt
⏭️ Skipping already translated file: SC_FR_244_2017.txt
⏭️ Skipping already translated file: SC_FR_284_2013.txt
⏭️ Skipping already translated file: SC_FR_29_2018.txt
⏭️ Skipping already translated file: SC_FR_233_2018.txt
⏭️ Skipping already translated file: SC_FR_430_2017.txt
⏭️ Skipping already translated file: SC_FR_24_2018.txt
⏭️ Skipping already translated file: SC_FR_290_2014.txt
⏭️ Skipping already translated file: SC_FR_253_2020.txt
⏭️ Skipping already translated file: SC_FR_422_2017.txt
⏭️ Skipping already translated file: SC_FR_498_2011.txt
⏭️ Skipping already translated file: SC_FR_129_2007.txt
⏭️ Skipping already translated file: SC_FR_212_2022.txt
⏭️ Skipping already translated file: SC_FR_79_2016.txt
⏭️ Skipping already translated file: SC_FR_351_2018_1.txt
⏭️ Skipping already translated file: SC_FR_859_2009.txt
⏭️ Skipping already translated file: SC_FR_444_2009.txt
⏭️ Skipping already translated file: SC_FR_27_201

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_52_2020_translated.txt
🧭 Total Sinhala blocks: 6

📄 Processing file: SC_CA_153_2013.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_153_2013_translated.txt
🧭 Total Sinhala blocks: 4

📄 Processing file: SC_CA_28_2008.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_28_2008_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_146_2013.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_146_2013_translated.txt
🧭 Total Sinhala blocks: 1

📄 Processing file: SC_CA_105_2018.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_105_2018_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_119_2014.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_119_20

Token indices sequence length is longer than the specified maximum sequence length for this model (1222 > 1024). Running this sequence through the model will result in indexing errors


✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_157_2019_translated.txt
🧭 Total Sinhala blocks: 3

📄 Processing file: SC_CA_120_2012.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_120_2012_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_62_2008.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_62_2008_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_184_2014.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_184_2014_translated.txt
🧭 Total Sinhala blocks: 2

📄 Processing file: SC_CA_10_2016.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_10_2016_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_85_2013.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_85_2013_

Token indices sequence length is longer than the specified maximum sequence length for this model (2008 > 1024). Running this sequence through the model will result in indexing errors


✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_104_2024_translated.txt
🧭 Total Sinhala blocks: 5

📄 Processing file: SC_CA_163_2019.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_163_2019_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_102_2010.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_102_2010_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_110_2018.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_110_2018_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_219_2016.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA_219_2016_translated.txt
🧭 Total Sinhala blocks: 0

📄 Processing file: SC_CA_104_105_2019.txt
✅ Translated file saved → /content/drive/MyDrive/Legal Summry/Translated/Civil Appeal/SC_CA

In [8]:
!zip -r "/content/Translated.zip" "/content/drive/MyDrive/Legal Summry/Translated"

  adding: content/drive/MyDrive/Legal Summry/Translated/ (stored 0%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/ (stored 0%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_139_2016_translated.txt (deflated 64%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_244_2017_translated.txt (deflated 72%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_284_2013_translated.txt (deflated 62%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_29_2018_translated.txt (deflated 69%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_233_2018_translated.txt (deflated 67%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_430_2017_translated.txt (deflated 48%)
  adding: content/drive/MyDrive/Legal Summry/Translated/Fundamental Rights/SC_FR_24_2018_translated.txt (deflated 74%)
  add